In [44]:
import mlflow
import dagshub
import scipy.sparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
import xgboost
from sklearn.naive_bayes import GaussianNB 
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import RandomOverSampler
sns.set(style='whitegrid')

import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

## Import Dataset

In [7]:
df = pd.read_csv("data.csv")
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


* id: Unique ID for the customer
* Gender: Gender of the customer
* Age: Age of the customer
* Driving_License: [0 : Customer does not have DL, 1 : Customer already has DL]
* Region_Code: Unique code for the region of the customer
* Previously_Insured: [1 : Customer already has Vehicle Insurance, 0 : Customer doesn't have Vehicle Insurance]
* Vehicle_Age: Age of the Vehicle
* Vehicle_Damage: [1 : Customer got his/her vehicle damaged in the past. 0 : Customer didn't get his/her vehicle damaged in the past.]
* Annual_Premium: The amount customer needs to pay as premium in the year
* Policy_Sales_Channel: Anonymized Code for the channel of outreaching to the customer ie. Different Agents, Over Mail, Over Phone, In Person, etc.
* Vintage: Number of Days, Customer has been associated with the company
* Response: [1 : Customer is interested, 0 : Customer is not interested]

In [8]:
# Checking the target distribution
print('Original class distribution:')
print(df['Response'].value_counts(normalize=True))

# Performing Stratified Sampling (10% of data)
X = df.drop('Response', axis=1)
y = df['Response']

# train_test_split for a stratified hold-out.
X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=0.1,
    random_state=42,
    stratify=y
)

# Combining back into a single DataFrame for sample
df_sample = pd.concat([X_sample, y_sample], axis=1)
print(f'\nSampled dataset shape: {df_sample.shape}')
print('Sampled class distribution:')
print(df_sample['Response'].value_counts(normalize=True))

Original class distribution:
Response
0    0.877437
1    0.122563
Name: proportion, dtype: float64

Sampled dataset shape: (38110, 12)
Sampled class distribution:
Response
0    0.877434
1    0.122566
Name: proportion, dtype: float64


## EDA

In [9]:
df_sample.shape

(38110, 12)

In [10]:
# checking for null values
df_sample.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [11]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38110 entries, 115711 to 133003
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    38110 non-null  int64  
 1   Gender                38110 non-null  object 
 2   Age                   38110 non-null  int64  
 3   Driving_License       38110 non-null  int64  
 4   Region_Code           38110 non-null  float64
 5   Previously_Insured    38110 non-null  int64  
 6   Vehicle_Age           38110 non-null  object 
 7   Vehicle_Damage        38110 non-null  object 
 8   Annual_Premium        38110 non-null  float64
 9   Policy_Sales_Channel  38110 non-null  float64
 10  Vintage               38110 non-null  int64  
 11  Response              38110 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 3.8+ MB


In [12]:
df_sample.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
count,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000
mean,191145.689976,38.835161,0.997927,26.338231,0.456940,30450.433482,112.202519,154.740698,0.122566
std,110199.646422,15.522027,0.045483,13.243016,0.498149,16833.200136,54.123557,83.524551,0.327943
min,5.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000,0.000000
25%,96059.750000,25.000000,1.000000,15.000000,0.000000,24259.500000,29.000000,82.000000,0.000000
50%,191443.500000,36.000000,1.000000,28.000000,0.000000,31619.000000,136.000000,155.000000,0.000000
75%,287265.250000,50.000000,1.000000,35.000000,1.000000,39384.750000,152.000000,227.000000,0.000000
max,381086.000000,85.000000,1.000000,52.000000,1.000000,346982.000000,163.000000,299.000000,1.000000


In [13]:
# checking distribution for target column
df_sample['Response'].value_counts()

Response
0    33439
1     4671
Name: count, dtype: int64

## Data Preprocessing

In [14]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38110 entries, 115711 to 133003
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    38110 non-null  int64  
 1   Gender                38110 non-null  object 
 2   Age                   38110 non-null  int64  
 3   Driving_License       38110 non-null  int64  
 4   Region_Code           38110 non-null  float64
 5   Previously_Insured    38110 non-null  int64  
 6   Vehicle_Age           38110 non-null  object 
 7   Vehicle_Damage        38110 non-null  object 
 8   Annual_Premium        38110 non-null  float64
 9   Policy_Sales_Channel  38110 non-null  float64
 10  Vintage               38110 non-null  int64  
 11  Response              38110 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 3.8+ MB


In [15]:
num_feat = ['Age','Vintage']
cat_feat = ['Gender', 'Driving_License', 'Previously_Insured', 'Vehicle_Age_lt_1_Year',
'Vehicle_Age_gt_2_Years','Vehicle_Damage_Yes','Region_Code','Policy_Sales_Channel']

In [16]:
df_sample['Gender'] = df_sample['Gender'].map({'Female': 0, 'Male': 1})
df_sample.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
115711,115712,1,43,1,46.0,0,1-2 Year,Yes,51672.0,11.0,38,1
334515,334516,1,24,1,50.0,0,< 1 Year,Yes,45236.0,152.0,109,0


In [17]:
for col in df.columns:
    print(f"{col} >> {df_sample[col].dtype}")

id >> int64
Gender >> int64
Age >> int64
Driving_License >> int64
Region_Code >> float64
Previously_Insured >> int64
Vehicle_Age >> object
Vehicle_Damage >> object
Annual_Premium >> float64
Policy_Sales_Channel >> float64
Vintage >> int64
Response >> int64


In [18]:
# creating dummy cols for categorical features

df_sample=pd.get_dummies(df_sample,drop_first=True)
df_sample.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Vehicle_Age_< 1 Year,Vehicle_Age_> 2 Years,Vehicle_Damage_Yes
115711,115712,1,43,1,46.0,0,51672.0,11.0,38,1,False,False,True
334515,334516,1,24,1,50.0,0,45236.0,152.0,109,0,True,False,True


In [19]:
for col in df_sample.columns:
    print(f"{col} >> {df_sample[col].dtype}")

id >> int64
Gender >> int64
Age >> int64
Driving_License >> int64
Region_Code >> float64
Previously_Insured >> int64
Annual_Premium >> float64
Policy_Sales_Channel >> float64
Vintage >> int64
Response >> int64
Vehicle_Age_< 1 Year >> bool
Vehicle_Age_> 2 Years >> bool
Vehicle_Damage_Yes >> bool


In [20]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38110 entries, 115711 to 133003
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     38110 non-null  int64  
 1   Gender                 38110 non-null  int64  
 2   Age                    38110 non-null  int64  
 3   Driving_License        38110 non-null  int64  
 4   Region_Code            38110 non-null  float64
 5   Previously_Insured     38110 non-null  int64  
 6   Annual_Premium         38110 non-null  float64
 7   Policy_Sales_Channel   38110 non-null  float64
 8   Vintage                38110 non-null  int64  
 9   Response               38110 non-null  int64  
 10  Vehicle_Age_< 1 Year   38110 non-null  bool   
 11  Vehicle_Age_> 2 Years  38110 non-null  bool   
 12  Vehicle_Damage_Yes     38110 non-null  bool   
dtypes: bool(3), float64(3), int64(7)
memory usage: 3.3 MB


In [21]:
# cols renaming and keeping dtype as int

df_sample = df_sample.rename(columns={"Vehicle_Age_< 1 Year": "Vehicle_Age_lt_1_Year", "Vehicle_Age_> 2 Years": "Vehicle_Age_gt_2_Years"})
df_sample['Vehicle_Age_lt_1_Year'] = df_sample['Vehicle_Age_lt_1_Year'].astype('int')
df_sample['Vehicle_Age_gt_2_Years'] = df_sample['Vehicle_Age_gt_2_Years'].astype('int')
df_sample['Vehicle_Damage_Yes'] = df_sample['Vehicle_Damage_Yes'].astype('int')

In [22]:
df_sample.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Vehicle_Age_lt_1_Year,Vehicle_Age_gt_2_Years,Vehicle_Damage_Yes
115711,115712,1,43,1,46.0,0,51672.0,11.0,38,1,0,0,1
334515,334516,1,24,1,50.0,0,45236.0,152.0,109,0,1,0,1
336126,336127,1,46,1,28.0,0,28410.0,26.0,299,0,0,0,1
231948,231949,0,44,1,47.0,0,29655.0,124.0,168,0,0,0,1
260249,260250,1,25,1,32.0,0,26374.0,26.0,221,0,1,0,1


In [23]:
# scaling the data

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler

ss = StandardScaler()
df_sample[num_feat] = ss.fit_transform(df_sample[num_feat])


mm = MinMaxScaler()
df_sample[['Annual_Premium']] = mm.fit_transform(df_sample[['Annual_Premium']])

# also, dropping id col now
id=df_sample.id
df_sample=df_sample.drop('id',axis=1)

In [24]:
# train-test split

from sklearn.model_selection import train_test_split

train_target=df_sample['Response']
train=df_sample.drop(['Response'], axis = 1)
x_train,x_test,y_train,y_test = train_test_split(train,train_target, random_state = 0)

In [25]:
train_target

115711    1
334515    0
336126    0
231948    0
260249    0
         ..
27570     0
80782     1
361828    0
338487    0
133003    0
Name: Response, Length: 38110, dtype: int64

In [26]:
train.head(1)

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Vehicle_Age_lt_1_Year,Vehicle_Age_gt_2_Years,Vehicle_Damage_Yes
115711,1,0.268321,1,46.0,0,0.142418,11.0,-1.3977,0,0,1


In [27]:
df_sample.info(

)

<class 'pandas.core.frame.DataFrame'>
Index: 38110 entries, 115711 to 133003
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Gender                  38110 non-null  int64  
 1   Age                     38110 non-null  float64
 2   Driving_License         38110 non-null  int64  
 3   Region_Code             38110 non-null  float64
 4   Previously_Insured      38110 non-null  int64  
 5   Annual_Premium          38110 non-null  float64
 6   Policy_Sales_Channel    38110 non-null  float64
 7   Vintage                 38110 non-null  float64
 8   Response                38110 non-null  int64  
 9   Vehicle_Age_lt_1_Year   38110 non-null  int64  
 10  Vehicle_Age_gt_2_Years  38110 non-null  int64  
 11  Vehicle_Damage_Yes      38110 non-null  int64  
dtypes: float64(5), int64(7)
memory usage: 3.8 MB


In [28]:
didy

NameError: name 'didy' is not defined

In [29]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow')
dagshub.init(repo_owner='MANJESH-ctrl', repo_name='MLOPS_proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("RF_and_Target_resampling")

Accessing as MANJESH-ctrl

Initialized MLflow to track repo "MANJESH-ctrl/MLOPS_proj"

Repository MANJESH-ctrl/MLOPS_proj initialized!

2026/01/11 13:13:27 INFO mlflow.tracking.fluent: Experiment with name 'RF_and_Target_resampling' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/43ba4133c1c0428ca7ff8a38dfc34c22', creation_time=1768117408239, experiment_id='2', last_update_time=1768117408239, lifecycle_stage='active', name='RF_and_Target_resampling', tags={}>

In [45]:
ALGORITHMS = {
    'GaussianNB': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

def train_and_evaluate(df_sample):
    with mlflow.start_run(run_name="All Experiments") as parent_run:
        for algo_name, algorithm in ALGORITHMS.items():
            with mlflow.start_run(run_name=algo_name , nested=True) as child_run:
                    try:
                        train_target=df_sample['Response']
                        train=df_sample.drop(['Response'], axis = 1)
                        x_train,x_test,y_train,y_test = train_test_split(train,train_target, random_state=0)

                        # Log preprocessing parameters
                        mlflow.log_params({
                            "algorithm": algo_name,   
                        })

                        # Train model
                        model = algorithm
                        model.fit(x_train, y_train)

                        # Log model parameters
                        log_model_params(algo_name, model)

                        # Evaluate model
                        y_pred = model.predict(x_test)
                        metrics = {
                            "accuracy": accuracy_score(y_test, y_pred),
                            "precision": precision_score(y_test, y_pred),
                            "recall": recall_score(y_test, y_pred),
                            "f1_score": f1_score(y_test, y_pred)
                        }
                        mlflow.log_metrics(metrics)

                        # Log model
                        # mlflow.sklearn.log_model(model, "model")
                        input_example = x_test[:5] if not scipy.sparse.issparse(x_test) else x_test[:5].toarray()
                        mlflow.sklearn.log_model(model, "model", input_example=input_example)

                        print(f"Metrics: {metrics}")

                    except Exception as e:
                        print(f"Error in training {algo_name}: {e}")
                        mlflow.log_param("error", str(e))

def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'XGBoost':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'GradientBoosting':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
        params_to_log["max_depth"] = model.max_depth
    mlflow.log_params(params_to_log)
            
if __name__ == "__main__":
    # df = load_data(CONFIG["data_path"])
    train_and_evaluate(df_sample)
# train_and_evaluate(df_sample)      

2026/01/11 13:56:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.7082283795130143, 'precision': 0.27205071664829106, 'recall': 0.8765541740674956, 'f1_score': 0.41522928060580566}
🏃 View run GaussianNB at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/f4567f9794ef4c6b93f50483efcdd66c
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 13:57:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.8712216624685138, 'precision': 0.35014836795252224, 'recall': 0.10479573712255773, 'f1_score': 0.16131237183868763}
🏃 View run XGBoost at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/55b59cf1006e465587f241f51bb78051
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 13:57:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.8702770780856424, 'precision': 0.31666666666666665, 'recall': 0.08436944937833037, 'f1_score': 0.1332398316970547}
🏃 View run RandomForest at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/32b20d0cfb3d44098d973cc88cf7f4ba
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 13:58:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.8809823677581864, 'precision': 0.2777777777777778, 'recall': 0.004440497335701598, 'f1_score': 0.008741258741258742}
🏃 View run GradientBoosting at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/4fe172f674f64580b8ada7f6c2db1937
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/416a950448d04c92a28fdd9ff40fc4db
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


In [46]:
from imblearn.combine import SMOTEENN
from collections import Counter

# 1. Assuming you are working with the sampled `df_sample`
# Separate features and target again
X_dev = df_sample.drop('Response', axis=1)
y_dev = df_sample['Response']
print('Before resampling:', Counter(y_dev))

# 2. Apply SMOTEENN (SMOTE + Edited Nearest Neighbors)
# SMOTE generates synthetic samples for the minority class.
# ENN removes majority class samples that are "hard to classify" near minority samples.
smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto')
X_resampled, y_resampled = smote_enn.fit_resample(X_dev, y_dev)

print('After SMOTEENN resampling:', Counter(y_resampled))

# 3. Combine back into a DataFrame for the balanced development set
df_balanced = pd.concat([pd.DataFrame(X_resampled), pd.DataFrame(y_resampled)], axis=1)
df_balanced.columns = list(X_dev.columns) + ['Response']  # Restore column names

Before resampling: Counter({0: 33439, 1: 4671})
After SMOTEENN resampling: Counter({1: 27224, 0: 23695})


In [47]:
print({df_balanced.shape})

{(50919, 12)}


In [48]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y) from the balanced DataFrame
X_balanced = df_balanced.drop('Response', axis=1)
y_balanced = df_balanced['Response']

# Perform train-test split on the balanced data
X_train_balanced, X_test_balanced, y_train_balanced, y_test_balanced = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f'Shape of X_train_balanced: {X_train_balanced.shape}')
print(f'Shape of X_test_balanced: {X_test_balanced.shape}')
print(f'Shape of y_train_balanced: {y_train_balanced.shape}')
print(f'Shape of y_test_balanced: {y_test_balanced.shape}')

# Check class distribution in training and test sets after balancing
print('\nClass distribution in y_train_balanced:')
print(y_train_balanced.value_counts(normalize=True))
print('\nClass distribution in y_test_balanced:')
print(y_test_balanced.value_counts(normalize=True))

Shape of X_train_balanced: (40735, 11)
Shape of X_test_balanced: (10184, 11)
Shape of y_train_balanced: (40735,)
Shape of y_test_balanced: (10184,)

Class distribution in y_train_balanced:
Response
1    0.534651
0    0.465349
Name: proportion, dtype: float64

Class distribution in y_test_balanced:
Response
1    0.534662
0    0.465338
Name: proportion, dtype: float64


In [ ]:
ALGORITHMS = {
    'GaussianNB': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

def train_and_evaluate(df_balanced):
    with mlflow.start_run(run_name="All Experiments with SMOTEENN") as parent_run:
        for algo_name, algorithm in ALGORITHMS.items():
            with mlflow.start_run(run_name=algo_name , nested=True) as child_run:
                    try:
                        train_target=df_balanced['Response']
                        train=df_balanced.drop(['Response'], axis = 1)
                        X_train_balanced, X_test_balanced, y_train_balanced, y_test_balanced = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)
                                                                                                            
                        
                        # Log preprocessing parameters
                        mlflow.log_params({
                            "algorithm": algo_name,   
                        })

                        # Train model
                        model = algorithm
                        model.fit(X_train_balanced, y_train_balanced)

                        # Log model parameters
                        log_model_params(algo_name, model)

                        # Evaluate model
                        y_pred = model.predict(X_test_balanced)
                        metrics = {
                            "accuracy": accuracy_score(y_test_balanced, y_pred),
                            "precision": precision_score(y_test_balanced, y_pred),
                            "recall": recall_score(y_test_balanced, y_pred),
                            "f1_score": f1_score(y_test_balanced, y_pred)
                        }
                        mlflow.log_metrics(metrics)

                        # Log model
                        # mlflow.sklearn.log_model(model, "model")
                        input_example = X_test_balanced[:5] if not scipy.sparse.issparse(X_test_balanced) else X_test_balanced[:5].toarray()
                        mlflow.sklearn.log_model(model, "model", input_example=input_example)

                        print(f"Metrics: {metrics}")

                    except Exception as e:
                        print(f"Error in training {algo_name}: {e}")
                        mlflow.log_param("error", str(e))

def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'XGBoost':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'GradientBoosting':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
        params_to_log["max_depth"] = model.max_depth
    mlflow.log_params(params_to_log)
            
if __name__ == "__main__":
    # df = load_data(CONFIG["data_path"])
    train_and_evaluate(df_sample)
# train_and_evaluate(df_sample)      

2026/01/11 14:14:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.8904163393558523, 'precision': 0.8449402390438246, 'recall': 0.9737373737373738, 'f1_score': 0.904778156996587}
🏃 View run GaussianNB at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/462ea0dcd7b241168b0cdac66470e6dd
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 14:14:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.9485467399842891, 'precision': 0.9291819291819292, 'recall': 0.9783287419651056, 'f1_score': 0.9531222043299338}
🏃 View run XGBoost at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/045ecffdce9a4399ad67042abc6384b8
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 14:15:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.959446190102121, 'precision': 0.9415584415584416, 'recall': 0.9853076216712581, 'f1_score': 0.9629363726106076}
🏃 View run RandomForest at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/739ea65fdf1e43328f43363eaf4890aa
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


2026/01/11 14:16:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Metrics: {'accuracy': 0.9245875883739199, 'precision': 0.8964231225631463, 'recall': 0.9711662075298438, 'f1_score': 0.9322990126939351}
🏃 View run GradientBoosting at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/335483cad8ce4774b7bca7ac6af40d64
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2/runs/4c99f00108cd4147a56f645539770977
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/2


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

random_search = {'criterion': ['entropy', 'gini'],
               'max_depth': [2,3,4,5,6,7,10],
               'min_samples_leaf': [4, 6, 8],
               'min_samples_split': [5, 7,10],
               'n_estimators': [300]}

clf = RandomForestClassifier(class_weight="balanced")
model = RandomizedSearchCV(estimator = clf, param_distributions = random_search, n_iter = 10,
                               cv = 4, verbose= 1, random_state= 42, n_jobs = -1)
model.fit(X_train_balanced,y_train_balanced)

In [ ]:
best_params = model.best_params_

print("Best Hyperparameters:")
print(best_params)